# FL/OD Analysis
Fluorescence normalized by fl, grouped by light condition and intensity.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

# ── Load data ──────────────────────────────────────────────────
od_df = pd.read_csv('od.csv')
fl_df = pd.read_csv('fl.csv')

# ── Time → hours ───────────────────────────────────────────────
def to_hours(t):
    h, m, s = t.split(':')
    return int(h) + int(m)/60 + int(s)/3600

time_h = fl_df['Time'].apply(to_hours).values

conditions = {
    'Red':      ['A3','A4','A6'],
    'Green':    ['B3','B4','B6'],
    '05%':      ['C1','D1','E1'],
    '10%':      ['F1','G1','H1'],
    '15%':      ['C3','C4','C6'],
    '20%':      ['D3','D4','D6'],
    '40%':      ['E3','E4','E6'],
    '60%':      ['F3','F4','F6'],
    '80%':      ['G3','G4','G6'],
    'Dark':     ['H3','H4','H6']
}

NEG_WELLS = ['A1','B1']
OD_BLANK = 0.1

# ── Subtract blanks ────────────────────────────────────────────
all_wells = NEG_WELLS + [w for ws in conditions.values() for w in ws]
od    = od_df[all_wells].astype(float)
fl    = fl_df[all_wells].astype(float)
fl_od = fl / od
neg_fl_od = fl_od[NEG_WELLS].mean(axis=1).values

# ── Shared colour maps (used in all cells) ─────────────────────
green_labels = ['05%','10%','15%','20%']
green_colors = plt.cm.Greens(np.linspace(0.25, 0.95, len(green_labels)))
green_shades = dict(zip(green_labels, green_colors))

blue_labels = ['40%', '60%', '80%']
blue_colors = plt.cm.Blues(np.linspace(0.25, 0.95, len(blue_labels)))
blue_shades = dict(zip(blue_labels, blue_colors))

cond_shades = {'Red': 'red', 'Green': 'green', 'Dark': 'gray'}
cond_shades.update(green_shades)
cond_shades.update(blue_shades)

print('Setup complete.')


In [ ]:
# ── Cell 2: FL/OD plot ─────────────────────────────────────────
results = {}
for cond, wells in conditions.items():
    vals = fl_od[wells].values - neg_fl_od[:, np.newaxis]
    mean = vals.mean(axis=1)
    se   = vals.std(axis=1, ddof=1) / np.sqrt(vals.shape[1]) if vals.shape[1] > 1 else np.zeros(len(mean))
    results[cond] = (mean, se)

fig, ax = plt.subplots(figsize=(7.5, 4.5))
fig.suptitle('FL/OD by Condition', fontsize=13, fontweight='bold')

# All conditions (Red, Green, Dark, and every duty-cycle %) plotted
for cond, color in cond_shades.items():
    mean, se = results[cond]
    ax.plot(time_h, mean, color=color, linestyle='--', label=f'{cond} (n={len(conditions[cond])})', linewidth=1.5)
    ax.fill_between(time_h, mean - se, mean + se, color=color, alpha=0.2)

ax.set_xlim(0, 16)
ax.set_ylim(0,30000)
ax.set_xlabel('Time (hours)', fontsize=13)
ax.set_ylabel('FL / OD (a.u.)', fontsize=13)
handles, labels = ax.get_legend_handles_labels()
handles, labels = zip(*sorted(zip(handles, labels), key=lambda t: t[1]))
ax.legend(handles, labels, fontsize=10, loc='upper left', frameon=False)
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=13)

plt.tight_layout()
plt.savefig('fl_od_figure.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Cell 3: smoothing and dx/dt ────────────────────────
from statsmodels.nonparametric.smoothers_lowess import lowess

plotting_conditions = list(conditions.keys())
smooth_time_window = [0,8.5] 
smooth_window_indx = np.where((smooth_time_window[0] < time_h) & (time_h < smooth_time_window[1]))
time_smooth = time_h[smooth_window_indx]
for frac in [0.3]:
    smoothed_results = [lowess(results[c][0][smooth_window_indx], time_smooth, frac=frac, return_sorted=False) for c in plotting_conditions]
    avg_grads = np.gradient(smoothed_results, axis=1).mean(axis=1)
    np.savetxt('avg_grads', avg_grads)

    fig, axs = plt.subplots(1, 2, figsize=(12, 4.5))
    axs[0].plot(time_smooth, np.transpose(smoothed_results), '--', lw = 1.5)
    axs[0].set_xlim(*smooth_time_window)
    axs[0].set_ylim(0,30000)
    axs[0].set_title("Smoothed Results")
    axs[0].tick_params(labelsize=13)
    axs[0].set_xlabel("Time (hours)", fontsize=13)
    axs[0].set_ylabel('FL / OD (a.u.)', fontsize=13)

    axs[1].plot(plotting_conditions, avg_grads, 'o')
    axs[1].set_title("Average Gradient with smoothing fraction = {}".format(frac))
    axs[1].tick_params(labelsize=13)
    axs[1].set_xlabel("Condition", fontsize=13)
    axs[1].set_ylabel("Average Gradient", fontsize=13)

In [ ]:
# ── Cell 4: OD plot (individual wells) ────────────────────────
fig, ax = plt.subplots(figsize=(7.5, 4.5))
fig.suptitle(r'$OD_{600}$', fontsize=16, fontweight='bold')

seen = set()
for cond, color in cond_shades.items():
    for well in conditions[cond]:
        ax.plot(time_h, od[well].values-OD_BLANK, color=color, linestyle='--', linewidth=1,
                label=cond if cond not in seen else '_')
        seen.add(cond)

ax.plot(time_h, od[NEG_WELLS].values-OD_BLANK, color='black', linewidth=1, label='Neg Cont.')

ax.set_xlim(0, 16)
ax.set_ylim(0,1.1)
ax.set_xlabel('Time (hours)', fontsize=15)
ax.set_ylabel(r'$OD_{600}$', fontsize=15)
handles, labels = ax.get_legend_handles_labels()
handles, labels = zip(*sorted(zip(handles, labels), key=lambda t: t[1]))
ax.legend(handles, labels, fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=12)

plt.tight_layout()
plt.savefig('od_figure.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Cell 5: Raw FL plot (individual wells) ─────────────────────
fig, ax = plt.subplots(figsize=(7.5, 4.5))
fig.suptitle('Raw Fluorescence (mCherry)', fontsize=16, fontweight='bold')

seen = set()

# All conditions, from the shared colour map (see Cell 4 note).
for cond, color in cond_shades.items():
    for well in conditions[cond]:
        ax.plot(time_h, fl[well].values, color=color, linestyle='--', linewidth=1,
                label=cond if cond not in seen else '_')
        seen.add(cond)

ax.plot(time_h, fl[NEG_WELLS].values, color='black', linewidth=1, label='NC')

ax.set_xlim(0, 16)
ax.set_ylim(0,35000)
ax.set_xlabel('Time (hours)', fontsize=15)
ax.set_ylabel('FL (a.u.)', fontsize=15)
handles, labels = ax.get_legend_handles_labels()
handles, labels = zip(*sorted(zip(handles, labels), key=lambda t: t[1]))
ax.legend(handles, labels, fontsize=9, loc='upper left')
ax.grid(True, alpha=0.3)
ax.tick_params(labelsize=12)

plt.tight_layout()
plt.savefig('fl_figure.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Cell 6: OD by column (2x2 grid) ───────────────────────────
col_wells = {
    'Column 1': ['A1','B1','C1','D1','E1','F1','G1','H1'],
    'Column 3': ['A3','B3','C3','D3','E3','F3','G3','H3'],
    'Column 4': ['A4','B4','C4','D4','E4','F4','G4','H4'],
    'Column 6': ['A6','B6','C6','D6','E6','F6','G6','H6'],
}

# Build well_style from conditions dict
well_style = {}
for cond, wells in conditions.items():
    color = cond_shades[cond]
    for w in wells:
        well_style[w] = (color, cond)

fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
fig.suptitle(r'$OD_{600}$ by Column', fontsize=14, fontweight='bold')

for ax, (col_name, wells) in zip(axes.flatten(), col_wells.items()):
    seen = set()
    for well in wells:
        if well not in well_style:
            continue
        color, label = well_style[well]
        ax.plot(time_h, od[well].values-OD_BLANK, color=color, linewidth=1.5,
                label=label if label not in seen else '_')
        seen.add(label)
    ax.set_xlim(0, 16); ax.set_ylim(0,1.1)
    ax.set_title(col_name, fontsize=12)
    ax.set_xlabel('Time (hours)', fontsize=11)
    handles, labels = ax.get_legend_handles_labels()
    handles, labels = zip(*sorted(zip(handles, labels), key=lambda t: t[1]))
    ax.legend(handles, labels, fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

axes[0,0].set_ylabel(r'$OD_{600}$', fontsize=12)
axes[1,0].set_ylabel(r'$OD_{600}$', fontsize=12)
plt.tight_layout()
plt.subplots_adjust(hspace=0.4)
# plt.savefig('od_by_column.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Cell 7: FL by column (2x2 grid) ───────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
fig.suptitle('FL by Column', fontsize=14, fontweight='bold')

for ax, (col_name, wells) in zip(axes.flatten(), col_wells.items()):
    seen = set()
    for well in wells:
        if well not in well_style:
            continue
        color, label = well_style[well]
        ax.plot(time_h, fl[well].values, color=color, linewidth=1.5,
                label=label if label not in seen else '_')
        seen.add(label)
    ax.set_xlim(0, 16); ax.set_ylim(0,40000)
    ax.set_title(col_name, fontsize=12)
    ax.set_xlabel('Time (hours)', fontsize=11)
    handles, labels = ax.get_legend_handles_labels()
    handles, labels = zip(*sorted(zip(handles, labels), key=lambda t: t[1]))
    ax.legend(handles, labels, fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

axes[0,0].set_ylabel('FL (a.u.)', fontsize=12)
axes[1,0].set_ylabel('FL (a.u.)', fontsize=12)
plt.tight_layout()
plt.subplots_adjust(hspace=0.4)
# plt.savefig('fl_by_column.png', dpi=150, bbox_inches='tight')
plt.show()


In [ ]:
# ── Cell 8: FL_OD by column (2x2 grid) ───────────────────────────
fig, axes = plt.subplots(2, 2, figsize=(14, 10), sharey=True)
fig.suptitle('FL/OD by Column', fontsize=14, fontweight='bold')

for ax, (col_name, wells) in zip(axes.flatten(), col_wells.items()):
    seen = set()
    for well in wells:
        if well not in well_style:
            continue
        color, label = well_style[well]
        ax.plot(time_h, fl_od[well].values, color=color, linewidth=1.5,
                label=label if label not in seen else '_')
        seen.add(label)
    ax.set_xlim(0, 16); ax.set_ylim(0,40000)
    ax.set_title(col_name, fontsize=12)
    ax.set_xlabel('Time (hours)', fontsize=11)
    handles, labels = ax.get_legend_handles_labels()
    handles, labels = zip(*sorted(zip(handles, labels), key=lambda t: t[1]))
    ax.legend(handles, labels, fontsize=9, loc='upper left')
    ax.grid(True, alpha=0.3)

axes[0,0].set_ylabel('FL/OD (a.u.)', fontsize=12)
axes[1,0].set_ylabel('FL/OD (a.u.)', fontsize=12)
plt.tight_layout()
plt.subplots_adjust(hspace=0.4)
plt.savefig('fl_od_by_column.png', dpi=150, bbox_inches='tight')
plt.show()
